#Emergent Models, trainer

First cell: general setup;
choose your drive folder


In [ ]:
# =========================
# Cell 1 — Colab setup + Google Drive
# =========================
!pip install taichi["cuda"]

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import csv
import numpy as np
from numpy.random import default_rng
import taichi as ti

# Change this if you want a different root folder on Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/EM43")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT.resolve())



In [ ]:

# =========================
# Cell 2 — Core utilities
# =========================
def lut_idx4(l: int, c: int, r: int) -> int:
    return (l * 4 + c) * 4 + r


IDX_000 = lut_idx4(0, 0, 0)  # -> 0
IDX_020 = lut_idx4(0, 2, 0)  # -> 2
IDX_200 = lut_idx4(2, 0, 0)  # -> 0
IDX_002 = lut_idx4(0, 0, 2)  # -> 0


def enforce_immutables_full(rule_full: np.ndarray) -> None:
    """
    Enforce immutable LUT outputs on a (..., 64) rule array in place.
    """
    if rule_full.shape[-1] != 64:
        raise ValueError(f"rule_full last dimension must be 64, got {rule_full.shape}")
    rule_full[..., IDX_000] = 0
    rule_full[..., IDX_020] = 2
    rule_full[..., IDX_200] = 0
    rule_full[..., IDX_002] = 0


def random_rule_full(rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, 4, size=(64,), dtype=np.uint8)


def random_prog(
    rng: np.random.Generator,
    Lp: int,
    probs: np.ndarray = np.array([0.55, 0.30, 0.12, 0.03], dtype=np.float64),
) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum()
    return rng.choice([0, 1, 2, 3], size=(Lp,), p=probs).astype(np.uint8)


def write_csv_rows(path: Path, header: list[str], rows: list[list]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def build_single_input_task(
    *,
    Le: int,
    inputs: list[int] | range,
    target_fn,
):
    """
    Positional single-marker encoding.
    Input x is encoded by placing a 2 at position x.
    """
    inps = list(inputs)
    M = len(inps)

    enc = np.zeros((M, Le), dtype=np.uint8)
    tgt = np.empty((M,), dtype=np.int32)

    for i, x in enumerate(inps):
        if not (0 <= x < Le):
            raise ValueError(f"Input x={x} is out of encoding range Le={Le}")
        y = int(target_fn(x))
        enc[i, x] = 2
        tgt[i] = y

    if np.any(tgt < 0):
        raise ValueError("Negative targets are not supported by this positional decoder")
    if np.any(tgt >= Le):
        raise ValueError(f"Some targets exceed decoder range [0, {Le - 1}]")

    return enc, tgt


def build_two_input_task(
    *,
    Le: int,
    pairs: list[tuple[int, int]],
    target_fn,
):
    """
    Positional two-marker encoding.
    Each pair (a, b) is encoded by placing 2 at positions a and b.
    Note: if a == b, the two markers collapse to one cell.
    """
    M = len(pairs)

    enc = np.zeros((M, Le), dtype=np.uint8)
    tgt = np.empty((M,), dtype=np.int32)

    for i, (a, b) in enumerate(pairs):
        if not (0 <= a < Le):
            raise ValueError(f"a={a} is out of encoding range Le={Le}")
        if not (0 <= b < Le):
            raise ValueError(f"b={b} is out of encoding range Le={Le}")

        y = int(target_fn(a, b))
        enc[i, a] = 2
        enc[i, b] = 2
        tgt[i] = y

    if np.any(tgt < 0):
        raise ValueError("Negative targets are not supported by this positional decoder")
    if np.any(tgt >= Le):
        raise ValueError(f"Some targets exceed decoder range [0, {Le - 1}]")

    return enc, tgt


# Taichi should be initialized once per runtime
_TI_INITIALIZED = False

def init_taichi_once(device_memory_fraction: float = 0.9):
    global _TI_INITIALIZED
    if not _TI_INITIALIZED:
        ti.init(arch=ti.gpu, device_memory_fraction=device_memory_fraction)
        _TI_INITIALIZED = True
        print("Taichi initialized on GPU.")


# =========================
#GPU simulator
# =========================
@ti.data_oriented
class EM43Model:
    """
    1D radius-1, 4-state CA model with:
      - full 64-entry LUT
      - immutable LUT entries enforced externally
      - input/output symbol = 2
      - halting/control symbol = 3
      - halting when control is at least half of live cells
      - invalid if:
          * touches right boundary at any time
          * no output 2 exists at halt
          * fails to halt within max_steps
      - invalid sample uses fake decode = -invalid_penalty

    Batch layout:
      rules_full : (I, P_i, 64)
      progs      : (I, P_i, Lp)
      encodings  : (M, Le)
      targets    : (M,)

    Returns:
      mae      : (I, P_i)
      fitness  : (I, P_i)
    """

    def __init__(
        self,
        Lp: int,
        Le: int,
        *,
        islands: int,
        pop_per_island: int,
        Ls: int = 4,
        device_memory_fraction: float = 0.9,
    ):
        init_taichi_once(device_memory_fraction=device_memory_fraction)

        self.I = int(islands)
        self.P_i = int(pop_per_island)
        self.B = self.I * self.P_i

        self.Lp = int(Lp)
        self.Le = int(Le)
        self.Ls = int(Ls)

        self.T = self.Lp + self.Ls + self.Le
        self.Tp = self.T + 4  # 2-cell halo each side

        self.ca = ti.field(dtype=ti.u8, shape=(2, self.B, self.Tp))
        self.rule = ti.field(dtype=ti.u8, shape=(self.B, 64))
        self.prog = ti.field(dtype=ti.u8, shape=(self.B, self.Lp))
        self.enc = ti.field(dtype=ti.u8, shape=(self.Le,))

        self.live = ti.field(dtype=ti.i32, shape=(self.B,))
        self.ctrl = ti.field(dtype=ti.i32, shape=(self.B,))
        self.halt = ti.field(dtype=ti.i32, shape=(self.B,))
        self.invalid = ti.field(dtype=ti.i32, shape=(self.B,))
        self.done = ti.field(dtype=ti.i32, shape=(self.B,))

        self.res = ti.field(dtype=ti.i32, shape=(self.B,))
        self.err = ti.field(dtype=ti.i32, shape=(self.B,))

        self.bufid = ti.field(dtype=ti.i32, shape=())
        self.acc_target = ti.field(dtype=ti.i32, shape=())
        self.decode_enc_start = ti.field(dtype=ti.i32, shape=())
        self.invalid_penalty = ti.field(dtype=ti.i32, shape=())
        self.all_done_flag = ti.field(dtype=ti.i32, shape=())

    @ti.kernel
    def _reset_err(self):
        for b in range(self.B):
            self.err[b] = 0

    @ti.kernel
    def _accumulate(self):
        tgt = self.acc_target[None]
        for b in range(self.B):
            self.err[b] += ti.abs(self.res[b] - tgt)

    @ti.kernel
    def _all_done_kernel(self):
        total = 0
        for b in range(self.B):
            total += self.done[b]
        self.all_done_flag[None] = 1 if total == self.B else 0

    def _all_done(self) -> bool:
        self._all_done_kernel()
        return bool(self.all_done_flag[None])

    @ti.kernel
    def _init(self):
        right_boundary = self.T + 1  # last inner tape cell

        for b, j in ti.ndrange(self.B, self.Tp):
            self.ca[0, b, j] = 0
            self.ca[1, b, j] = 0

        for b, j in ti.ndrange(self.B, self.Lp):
            self.ca[0, b, j + 2] = self.prog[b, j]

        enc_start = 2 + self.Lp + self.Ls
        for b, j in ti.ndrange(self.B, self.Le):
            self.ca[0, b, enc_start + j] = self.enc[j]

        for b in range(self.B):
            live = 0
            ctrl = 0
            for i in range(self.T):
                v = self.ca[0, b, i + 2]
                if v != 0:
                    live += 1
                    if v == 3:
                        ctrl += 1

            self.live[b] = live
            self.ctrl[b] = ctrl
            self.halt[b] = 0
            self.invalid[b] = 0
            self.done[b] = 0
            self.res[b] = 0

            if self.ca[0, b, right_boundary] != 0:
                self.invalid[b] = 1
                self.done[b] = 1
            elif live > 0 and ctrl * 2 >= live:
                self.halt[b] = 1
                self.done[b] = 1

        self.bufid[None] = 0

    @ti.kernel
    def _step(self):
        src = self.bufid[None]
        dst = src ^ 1

        for b, i in ti.ndrange(self.B, self.T):
            j = i + 2
            old = self.ca[src, b, j]

            if self.done[b] == 1:
                self.ca[dst, b, j] = old
            else:
                l1 = ti.cast(self.ca[src, b, j - 1], ti.i32)
                c = ti.cast(old, ti.i32)
                r1 = ti.cast(self.ca[src, b, j + 1], ti.i32)
                idx = (l1 * 4 + c) * 4 + r1
                new = self.rule[b, idx]

                self.ca[dst, b, j] = new

                if old != 0:
                    ti.atomic_sub(self.live[b], 1)
                    if old == 3:
                        ti.atomic_sub(self.ctrl[b], 1)

                if new != 0:
                    ti.atomic_add(self.live[b], 1)
                    if new == 3:
                        ti.atomic_add(self.ctrl[b], 1)

        self.bufid[None] = dst

    @ti.kernel
    def _update_status_after_step(self):
        buf = self.bufid[None]
        right_boundary = self.T + 1

        for b in range(self.B):
            if self.done[b] == 0:
                if self.ca[buf, b, right_boundary] != 0:
                    self.invalid[b] = 1
                    self.done[b] = 1
                else:
                    live = self.live[b]
                    ctrl = self.ctrl[b]
                    if live > 0 and ctrl * 2 >= live:
                        self.halt[b] = 1
                        self.done[b] = 1

    @ti.kernel
    def _mark_unfinished_invalid(self):
        for b in range(self.B):
            if self.done[b] == 0:
                self.invalid[b] = 1
                self.done[b] = 1

    @ti.kernel
    def _decode(self):
        buf = self.bufid[None]
        es = self.decode_enc_start[None]
        fake_invalid_pred = -self.invalid_penalty[None]

        for b in range(self.B):
            if self.invalid[b] == 1:
                self.res[b] = fake_invalid_pred
            elif self.halt[b] == 1:
                p = -1
                for k in range(self.Le):
                    j = es + self.Le - 1 - k
                    if self.ca[buf, b, j] == 2:
                        p = j - es
                        break

                if p == -1:
                    self.invalid[b] = 1
                    self.done[b] = 1
                    self.res[b] = fake_invalid_pred
                else:
                    self.res[b] = p
            else:
                # Taichi kernels cannot raise; mark impossible state as invalid
                self.invalid[b] = 1
                self.done[b] = 1
                self.res[b] = fake_invalid_pred

    def evaluate_batch(
        self,
        rules_full: np.ndarray,
        progs: np.ndarray,
        encodings: np.ndarray,
        targets: np.ndarray,
        *,
        max_steps: int = 300,
        lambda_p: float = 0.0,
        invalid_penalty: int = 1000,
    ):
        """
        Returns:
          mae      : (I, P_i) float32
          fitness  : (I, P_i) float32
        """
        if rules_full.shape != (self.I, self.P_i, 64):
            raise ValueError(f"rules_full shape mismatch: expected {(self.I, self.P_i, 64)}, got {rules_full.shape}")
        if progs.shape != (self.I, self.P_i, self.Lp):
            raise ValueError(f"progs shape mismatch: expected {(self.I, self.P_i, self.Lp)}, got {progs.shape}")
        if encodings.ndim != 2 or encodings.shape[1] != self.Le:
            raise ValueError(f"encodings shape mismatch: expected (*, {self.Le}), got {encodings.shape}")
        if targets.ndim != 1 or targets.shape[0] != encodings.shape[0]:
            raise ValueError(f"targets shape mismatch: expected ({encodings.shape[0]},), got {targets.shape}")
        if invalid_penalty <= 0:
            raise ValueError(f"invalid_penalty must be positive, got {invalid_penalty}")
        if max_steps <= 0:
            raise ValueError(f"max_steps must be positive, got {max_steps}")

        M = encodings.shape[0]

        rules_flat = rules_full.reshape(self.B, 64).copy()
        progs_flat = progs.reshape(self.B, self.Lp).copy()
        enforce_immutables_full(rules_flat)

        self.rule.from_numpy(rules_flat)
        self.prog.from_numpy(progs_flat)

        density = (np.count_nonzero(progs_flat, axis=1).astype(np.float32) / float(self.Lp))

        self._reset_err()
        ti.sync()

        enc_start = 2 + self.Lp + self.Ls
        self.decode_enc_start[None] = enc_start
        self.invalid_penalty[None] = int(invalid_penalty)

        for k in range(M):
            self.enc.from_numpy(encodings[k])
            self._init()
            ti.sync()

            for _ in range(int(max_steps)):
                if self._all_done():
                    break
                self._step()
                self._update_status_after_step()
                ti.sync()

            self._mark_unfinished_invalid()
            self._decode()

            self.acc_target[None] = int(targets[k])
            self._accumulate()
            ti.sync()

        mae_flat = self.err.to_numpy().astype(np.float32) / float(M)
        fitness_flat = -mae_flat - float(lambda_p) * density

        mae = mae_flat.reshape(self.I, self.P_i).astype(np.float32)
        fitness = fitness_flat.reshape(self.I, self.P_i).astype(np.float32)
        return mae, fitness


# =========================
#Island GA
# =========================
class EM43GAIsland:
    """
    Island GA with:
      1) elites
      2) cross-island immigrants (sampled from full elite blocks of other islands)
      3) random immigrants
      4) children filling the remaining slots

    Selection for children:
      - one tournament per child
      - tournament winner = parent 1
      - parent 2 sampled from the parent-1 winner pool

    Tracks the true best-ever genome seen across training.
    """

    def __init__(
        self,
        Lp: int,
        Le: int,
        *,
        islands: int = 20,
        pop_per_island: int = 3000,
        generations: int = 300,
        elite_frac: float = 0.03,
        cross_imm_frac: float = 0.00,
        cross_interval: int = 10,
        rand_imm_frac: float = 0.02,
        tourney_k: int = 16,
        mut_rule: float = 0.02,
        mut_prog: float = 0.04,
        cross_rule_q: float = 0.20,
        cross_prog_q: float = 0.20,
        lambda_p: float = 0.001,
        invalid_penalty: int = 1000,
        temp: float = 1.3,
        max_steps: int = 500,
        gpu_mem_frac: float = 0.9,
        prog_init_probs: np.ndarray | None = None,
    ):
        self.rng = default_rng()

        self.I = int(islands)
        self.P_i = int(pop_per_island)
        self.P = self.I * self.P_i
        self.G = int(generations)

        self.Lp = int(Lp)
        self.Le = int(Le)

        if self.I <= 0 or self.P_i <= 0 or self.G <= 0:
            raise ValueError("islands, pop_per_island, and generations must be positive")
        if not (0.0 <= elite_frac <= 1.0):
            raise ValueError("elite_frac must be in [0,1]")
        if not (0.0 <= cross_imm_frac <= 1.0):
            raise ValueError("cross_imm_frac must be in [0,1]")
        if not (0.0 <= rand_imm_frac <= 1.0):
            raise ValueError("rand_imm_frac must be in [0,1]")
        if tourney_k < 1 or tourney_k > self.P_i:
            raise ValueError(f"tourney_k must be in [1, {self.P_i}]")
        if temp <= 0.0:
            raise ValueError("temp must be > 0")
        if invalid_penalty <= 0:
            raise ValueError("invalid_penalty must be > 0")
        if max_steps <= 0:
            raise ValueError("max_steps must be > 0")

        self.E_i = max(1, int(elite_frac * self.P_i))
        self.CI_i = int(cross_imm_frac * self.P_i)
        self.RI_i = int(rand_imm_frac * self.P_i)

        self.cross_interval = int(cross_interval)

        if self.E_i + self.CI_i + self.RI_i > self.P_i:
            raise ValueError(
                f"elite + cross_imm + rand_imm exceeds island size: "
                f"{self.E_i} + {self.CI_i} + {self.RI_i} > {self.P_i}"
            )

        self.K = int(tourney_k)
        self.m_r = float(mut_rule)
        self.m_p = float(mut_prog)
        self.q_r = float(cross_rule_q)
        self.q_p = float(cross_prog_q)
        self.lambda_p = float(lambda_p)
        self.invalid_penalty = int(invalid_penalty)
        self.temp = float(temp)
        self.max_steps = int(max_steps)

        self.sim = EM43Model(
            Lp=self.Lp,
            Le=self.Le,
            islands=self.I,
            pop_per_island=self.P_i,
            Ls=4,
            device_memory_fraction=gpu_mem_frac,
        )

        init_probs = (
            np.array([0.55, 0.30, 0.12, 0.03], dtype=np.float64)
            if prog_init_probs is None
            else np.asarray(prog_init_probs, dtype=np.float64)
        )
        init_probs = init_probs / init_probs.sum()

        self.rules = np.empty((self.I, self.P_i, 64), dtype=np.uint8)
        self.progs = np.empty((self.I, self.P_i, self.Lp), dtype=np.uint8)

        for i in range(self.I):
            for j in range(self.P_i):
                self.rules[i, j] = random_rule_full(self.rng)
                self.progs[i, j] = random_prog(self.rng, self.Lp, probs=init_probs)

        enforce_immutables_full(self.rules.reshape(self.P, 64))

        self.enc = None
        self.tgt = None
        self._paper = []

    def set_task(self, encodings: np.ndarray, targets: np.ndarray):
        encodings = np.asarray(encodings, dtype=np.uint8)
        targets = np.asarray(targets, dtype=np.int32)

        if encodings.ndim != 2 or encodings.shape[1] != self.Le:
            raise ValueError(f"encodings must have shape (M, {self.Le}), got {encodings.shape}")
        if targets.ndim != 1 or targets.shape[0] != encodings.shape[0]:
            raise ValueError(f"targets must have shape ({encodings.shape[0]},), got {targets.shape}")

        self.enc = encodings
        self.tgt = targets

    def _sample_cross_island_immigrants(
        self,
        elite_rules: np.ndarray,
        elite_progs: np.ndarray,
        dest_island: int,
        n_cross: int,
    ):
        if n_cross == 0:
            return None, None
        if self.I < 2:
            raise RuntimeError("cross-island immigrants requested with only one island")
        if elite_rules.shape != (self.I, self.E_i, 64):
            raise RuntimeError(f"elite_rules shape mismatch: got {elite_rules.shape}")
        if elite_progs.shape != (self.I, self.E_i, self.Lp):
            raise RuntimeError(f"elite_progs shape mismatch: got {elite_progs.shape}")

        other = np.delete(np.arange(self.I), dest_island)
        if other.size == 0:
            raise RuntimeError("no source islands available for cross-island immigration")

        pool_rules = elite_rules[other].reshape(-1, 64)
        pool_progs = elite_progs[other].reshape(-1, self.Lp)

        if pool_rules.shape[0] == 0:
            raise RuntimeError("cross-island immigrant pool is empty")

        idx = self.rng.integers(0, pool_rules.shape[0], size=n_cross)
        return pool_rules[idx].copy(), pool_progs[idx].copy()

    def _make_children_for_island(
        self,
        island_idx: int,
        fit_i: np.ndarray,
        rules_i: np.ndarray,
        progs_i: np.ndarray,
        n_child: int,
    ):
        if n_child < 0:
            raise RuntimeError(f"n_child is negative for island {island_idx}: {n_child}")
        if n_child == 0:
            return (
                np.empty((0, 64), dtype=np.uint8),
                np.empty((0, self.Lp), dtype=np.uint8),
            )

        grp = self.rng.integers(0, self.P_i, size=(n_child, self.K))
        subf = fit_i[grp]

        sf = subf - subf.max(axis=1, keepdims=True)
        w = np.exp(sf / self.temp)
        w_sum = w.sum(axis=1, keepdims=True)
        if np.any(w_sum == 0):
            raise RuntimeError("softmax weights sum to zero during tournament selection")
        w /= w_sum

        u = self.rng.random((n_child, 1))
        cum = np.cumsum(w, axis=1)
        win_pos = (u < cum).argmax(axis=1)
        p1 = grp[np.arange(n_child), win_pos]

        if p1.shape != (n_child,):
            raise RuntimeError(f"parent-1 shape mismatch: expected {(n_child,)}, got {p1.shape}")

        p2 = self.rng.choice(p1, size=n_child, replace=True)
        if p2.shape != (n_child,):
            raise RuntimeError(f"parent-2 shape mismatch: expected {(n_child,)}, got {p2.shape}")

        child_r = rules_i[p1].copy()
        child_p = progs_i[p1].copy()

        if self.q_r > 0.0:
            mask_r = self.rng.random(child_r.shape) < self.q_r
            child_r[mask_r] = rules_i[p2][mask_r]

        if self.q_p > 0.0:
            mask_p = self.rng.random(child_p.shape) < self.q_p
            child_p[mask_p] = progs_i[p2][mask_p]

        if self.m_r > 0.0:
            mut_r = self.rng.random(child_r.shape) < self.m_r
            n = int(mut_r.sum())
            if n > 0:
                child_r[mut_r] = self.rng.integers(0, 4, size=n, dtype=np.uint8)

        if self.m_p > 0.0:
            mut_p = self.rng.random(child_p.shape) < self.m_p
            n = int(mut_p.sum())
            if n > 0:
                child_p[mut_p] = self.rng.choice(
                    [0, 1, 2, 3],
                    size=n,
                    p=[0.55, 0.30, 0.12, 0.03],
                ).astype(np.uint8)

        enforce_immutables_full(child_r)

        if child_r.shape != (n_child, 64):
            raise RuntimeError(f"child_r shape mismatch: expected {(n_child, 64)}, got {child_r.shape}")
        if child_p.shape != (n_child, self.Lp):
            raise RuntimeError(f"child_p shape mismatch: expected {(n_child, self.Lp)}, got {child_p.shape}")

        return child_r, child_p

    def evolve(self):
        if self.enc is None or self.tgt is None:
            raise RuntimeError("Call set_task() before evolve().")

        rules = self.rules
        progs = self.progs

        best_global_fit = -np.inf
        best_ever_rule = None
        best_ever_prog = None
        best_ever_mae = None
        best_ever_gen = None
        best_ever_row = None

        hist_best_fit = []
        self._paper = []

        for g in range(1, self.G + 1):
            mae, fit = self.sim.evaluate_batch(
                rules,
                progs,
                self.enc,
                self.tgt,
                max_steps=self.max_steps,
                lambda_p=self.lambda_p,
                invalid_penalty=self.invalid_penalty,
            )

            fit_flat = fit.reshape(-1)
            mae_flat = mae.reshape(-1)

            order = np.argsort(fit_flat)[::-1]
            fit_sorted = fit_flat[order]
            mae_sorted = mae_flat[order]

            top5_n = max(1, int(0.05 * fit_sorted.size))
            top5_fit = fit_sorted[:top5_n]
            top5_mae = mae_sorted[:top5_n]

            row = [
                g,
                float(mae_sorted[0]),
                float(fit_sorted[0]),
                int(top5_n),
                float(top5_mae.mean()),
                float(top5_mae.std(ddof=0)),
                float(top5_fit.mean()),
                float(top5_fit.std(ddof=0)),
                int(mae_flat.size),
                float(mae_flat.mean()),
                float(mae_flat.std(ddof=0)),
                float(fit_flat.mean()),
                float(fit_flat.std(ddof=0)),
            ]
            self._paper.append(row)

            current_best_flat_idx = int(np.argmax(fit_flat))
            current_best_fit = float(fit_flat[current_best_flat_idx])

            if current_best_fit > best_global_fit:
                ii, jj = np.unravel_index(current_best_flat_idx, fit.shape)
                best_global_fit = current_best_fit
                best_ever_rule = rules[ii, jj].copy()
                best_ever_prog = progs[ii, jj].copy()
                best_ever_mae = float(mae[ii, jj])
                best_ever_gen = int(g)
                best_ever_row = row.copy()

            hist_best_fit.append(float(best_global_fit))

            if g == 1 or g % 10 == 0:
                best_local_fit = fit.max(axis=1)
                mean_best_fit = float(best_local_fit.mean())
                std_best_fit = float(best_local_fit.std(ddof=0))
                cv = (std_best_fit / (abs(mean_best_fit) + 1e-12)) * 100.0
                shown = "  ".join(
                    f"{k + 1}:{best_local_fit[k]:.4f}" for k in range(min(10, self.I))
                )
                print("--------------------------------")
                print(f"Gen {g:4d}")
                print(f"fitness (top per island) -> {shown}")
                print(f"mean best fit={mean_best_fit:.4f}   std/mean={cv:.1f}%")
                print(f"global best fit so far={best_global_fit:.4f}")
                print("--------------------------------")

            # Stop here on the last evaluated generation.
            # This keeps the number of evaluated generations exactly equal to self.G.
            if g == self.G:
                break

            idx = np.argsort(fit, axis=1)[:, ::-1]
            sorted_rules = np.take_along_axis(rules, idx[:, :, None], axis=1)
            sorted_progs = np.take_along_axis(progs, idx[:, :, None], axis=1)
            sorted_fit = np.take_along_axis(fit, idx, axis=1)

            next_rules = np.empty_like(sorted_rules)
            next_progs = np.empty_like(sorted_progs)

            next_rules[:, :self.E_i] = sorted_rules[:, :self.E_i]
            next_progs[:, :self.E_i] = sorted_progs[:, :self.E_i]

            elite_source_rules = next_rules[:, :self.E_i].copy()
            elite_source_progs = next_progs[:, :self.E_i].copy()

            cross_active = (self.cross_interval > 0 and g % self.cross_interval == 0)
            n_cross = self.CI_i if cross_active else 0

            if n_cross > 0 and self.I < 2:
                raise RuntimeError("cross-island immigrants requested but only one island exists")

            for i in range(self.I):
                pos = 0

                # 1) elites
                pos += self.E_i

                # 2) cross-island immigrants
                if n_cross > 0:
                    cross_r, cross_p = self._sample_cross_island_immigrants(
                        elite_source_rules,
                        elite_source_progs,
                        dest_island=i,
                        n_cross=n_cross,
                    )
                    if cross_r.shape != (n_cross, 64):
                        raise RuntimeError(f"cross_r shape mismatch: expected {(n_cross, 64)}, got {cross_r.shape}")
                    if cross_p.shape != (n_cross, self.Lp):
                        raise RuntimeError(f"cross_p shape mismatch: expected {(n_cross, self.Lp)}, got {cross_p.shape}")
                    next_rules[i, pos:pos + n_cross] = cross_r
                    next_progs[i, pos:pos + n_cross] = cross_p
                    pos += n_cross

                # 3) random immigrants
                if self.RI_i > 0:
                    rr = self.rng.integers(0, 4, size=(self.RI_i, 64), dtype=np.uint8)
                    rp = self.rng.choice(
                        [0, 1, 2, 3],
                        size=(self.RI_i, self.Lp),
                        p=[0.4, 0.3, 0.2, 0.1],
                    ).astype(np.uint8)
                    enforce_immutables_full(rr)
                    next_rules[i, pos:pos + self.RI_i] = rr
                    next_progs[i, pos:pos + self.RI_i] = rp
                    pos += self.RI_i

                # 4) children
                n_child = self.P_i - pos
                if n_child < 0:
                    raise RuntimeError(
                        f"Negative child count for island {i}: "
                        f"P_i={self.P_i}, pos={pos}, n_child={n_child}"
                    )

                child_r, child_p = self._make_children_for_island(
                    island_idx=i,
                    fit_i=sorted_fit[i],
                    rules_i=sorted_rules[i],
                    progs_i=sorted_progs[i],
                    n_child=n_child,
                )

                if child_r.shape[0] != n_child or child_p.shape[0] != n_child:
                    raise RuntimeError(
                        f"Child count mismatch for island {i}: "
                        f"expected {n_child}, got rules {child_r.shape[0]}, progs {child_p.shape[0]}"
                    )

                next_rules[i, pos:] = child_r
                next_progs[i, pos:] = child_p

                if next_rules[i].shape != (self.P_i, 64):
                    raise RuntimeError(f"Final rule island shape mismatch at island {i}: {next_rules[i].shape}")
                if next_progs[i].shape != (self.P_i, self.Lp):
                    raise RuntimeError(f"Final prog island shape mismatch at island {i}: {next_progs[i].shape}")

            rules = next_rules
            progs = next_progs
            enforce_immutables_full(rules.reshape(self.P, 64))

        if best_ever_rule is None or best_ever_prog is None or best_ever_mae is None or best_ever_gen is None or best_ever_row is None:
            raise RuntimeError("No best-ever genome was recorded during evolve().")

        self.rules = rules
        self.progs = progs

        return (
            best_ever_rule,
            best_ever_prog,
            float(best_ever_mae),
            float(best_global_fit),
            hist_best_fit,
            int(best_ever_gen),
            best_ever_row,
        )

    @staticmethod
    def save_genome(
        rule_full,
        program,
        mae,
        fitness,
        *,
        best_gen: int,
        task_alias: str,
        Lp: int,
        run: int,
        path: str | Path,
    ):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(
            path,
            rule_full=rule_full,
            program=program,
            mae=np.float32(mae),
            fitness=np.float32(fitness),
            best_gen=np.int32(best_gen),
            task_alias=np.array(task_alias),
            Lp=np.int32(Lp),
            run=np.int32(run),
        )
        print("saved ->", path.resolve())



#1-input task setup
1. edit Lp, Le to set the maximum program length and encoding length, and max_T for the maximum computation time allowed during training
2. edit 'target_fn' to set the function you want to train on
3. edit genetic algorithm hyperparams
4. skip at the running cell

In [ ]:
task_alias = "mod2c"

# Number of independent stochastic runs
tot_runs = 2

# Save directory for this task
TASK_DIR = DRIVE_ROOT / task_alias
TASK_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Task-specific dimensions
# -------------------------
I, P_i = 20, 2000
Lp, Le = 10, 80
max_T=150

# -------------------------
# Build task
# -------------------------
# Example: plus1 on x in [0, 20]
x_min, x_max = 0, 20

enc, tgt = build_single_input_task(
    Le=Le,
    inputs=range(x_min, x_max),
    target_fn=lambda x: x*(x%2 + 1),
)

# -------------------------
# GA hyperparameters
# -------------------------
ga_kwargs = dict(
    Lp=Lp,
    Le=Le,
    islands=I,
    pop_per_island=P_i,
    generations=800,
    elite_frac=0.05,
    cross_imm_frac=0.01,
    cross_interval=30,
    rand_imm_frac=0.05,
    tourney_k=2,
    mut_rule=0.025,
    mut_prog=0.04,
    cross_rule_q=0.05,
    cross_prog_q=0.07,
    lambda_p=0.003,
    invalid_penalty=5,
    temp=0.3,
    max_steps=max_T,
    gpu_mem_frac=0.9,
)

print("Task alias:", task_alias)
print("Task dir:", TASK_DIR.resolve())
print("Lp:", Lp)
print("Le:", Le)
print("Dataset size:", len(tgt))
print("Total runs:", tot_runs)



# Setup for 2 input tasks
*Skip if running 1-input task*

In [ ]:

task_alias = "add_ab"
tot_runs = 5
TASK_DIR = DRIVE_ROOT / task_alias
TASK_DIR.mkdir(parents=True, exist_ok=True)

I, P_i = 100, 500
Lp, Le = 10, 100

a_min, a_max = 0, 8
b_min, b_max = 0, 8

pairs = []
for a in range(a_min, a_max + 1):
    for b in range(b_min, b_max + 1):
        pairs.append((a, b))

enc, tgt = build_two_input_task(
    Le=Le,
    pairs=pairs,
    target_fn=lambda a, b: a + b,
)

ga_kwargs = dict(
    Lp=Lp,
    Le=Le,
    islands=I,
    pop_per_island=P_i,
    generations=500,
    elite_frac=0.3,
    cross_imm_frac=0.005,
    cross_interval=100,
    rand_imm_frac=0.01,
    tourney_k=2,
    mut_rule=0.02,
    mut_prog=0.03,
    cross_rule_q=0.03,
    cross_prog_q=0.04,
    lambda_p=0.005,
    invalid_penalty=7,
    temp=0.1,
    max_steps=200,
    gpu_mem_frac=0.9,
)




# Run training

In [ ]:

trajectory_header = [
    "run",
    "gen",
    "best_mae",
    "best_fit",
    "top5_n",
    "top5_mae_avg",
    "top5_mae_std",
    "top5_fit_avg",
    "top5_fit_std",
    "all_n",
    "all_mae_avg",
    "all_mae_std",
    "all_fit_avg",
    "all_fit_std",
]

# Includes all requested metrics, plus small metadata
run_summary_header = [
    "run",
    "task_alias",
    "Lp",
    "gen",
    "best_mae",
    "best_fit",
    "top5_n",
    "top5_mae_avg",
    "top5_mae_std",
    "top5_fit_avg",
    "top5_fit_std",
    "all_n",
    "all_mae_avg",
    "all_mae_std",
    "all_fit_avg",
    "all_fit_std",
    "genome_file",
]

trajectory_rows = []
run_summary_rows = []

trajectory_csv = TASK_DIR / f"paper_{task_alias}_trajectory.csv"
run_summary_csv = TASK_DIR / f"paper_{task_alias}_runs.csv"

for run_id in range(1, tot_runs + 1):
    print()
    print("=" * 70)
    print(f"RUN {run_id}/{tot_runs} — task={task_alias}")
    print("=" * 70)

    ga = EM43GAIsland(**ga_kwargs)
    ga.set_task(enc, tgt)

    (
        best_rule,
        best_prog,
        best_mae,
        best_fit,
        hist_best_fit,
        best_gen,
        best_row,
    ) = ga.evolve()

    genome_filename = f"best_em43_{task_alias}_run_{run_id:03d}.npz"
    genome_path = TASK_DIR / genome_filename

    EM43GAIsland.save_genome(
        rule_full=best_rule,
        program=best_prog,
        mae=best_mae,
        fitness=best_fit,
        best_gen=best_gen,
        task_alias=task_alias,
        Lp=Lp,
        run=run_id,
        path=genome_path,
    )

    # Per-generation trajectory rows: prepend run id
    for row in ga._paper:
        trajectory_rows.append([run_id] + row)

    # Per-run summary row:
    # run, task_alias, Lp, [best-ever row...], genome_file
    run_summary_rows.append(
        [run_id, task_alias, Lp] + best_row + [genome_filename]
    )

    # Save incrementally after each run for safety
    write_csv_rows(trajectory_csv, trajectory_header, trajectory_rows)
    write_csv_rows(run_summary_csv, run_summary_header, run_summary_rows)

    print(f"Run {run_id} done.")
    print(f"Best-ever gen: {best_gen}")
    print(f"Best-ever MAE: {best_mae:.6f}")
    print(f"Best-ever fitness: {best_fit:.6f}")
    print(f"Genome file: {genome_path.resolve()}")

print()
print("All runs completed.")
print("Trajectory CSV ->", trajectory_csv.resolve())
print("Run summary CSV ->", run_summary_csv.resolve())
